# Aprendizado de Máquina — Lista prática 06

## Pré-processamento e *Pipelines*

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Três exercícios, na mesma ordem da aula, e cada um responde a uma pergunta:

1. **para quem** a escala das covariáveis importa?
2. **quanto** custa padronizar na hora errada?
3. como se monta o objeto que impede a hora errada — e como se busca um
   hiperparâmetro escondido dentro dele?

As sementes aqui são diferentes das do laboratório, de propósito: os números não
vão bater com os de lá, e não adianta copiá-los.

Cada lacuna está marcada com `...` e tem uma letra no comentário ao lado — `# (a)`,
`# (b)` — para você saber quantas são. Substitua **todas** antes de rodar a célula.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd

import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.datasets import make_regression
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — quem muda quando a régua muda

Cinco métodos, um mesmo problema, e uma única coluna anotada em outra unidade: 500
vezes maior, como quem troca metros por milímetros.

A informação nos dados é exatamente a mesma nos dois casos. A coluna diz a mesma
coisa sobre o mundo, só está escrita em outro número. Em princípio, portanto,
nenhum método deveria mudar de resposta.

**Antes de preencher as lacunas, faça a previsão.** A pergunta que decide é sempre
a mesma: *o método soma coisas vindas de colunas diferentes?* Se soma, as unidades
precisam ser comparáveis, e trocar uma delas estraga a comparação. Se não soma, a
unidade é irrelevante. Percorra os cinco métodos da célula com essa pergunta na
cabeça e anote o palpite.

As três lacunas testam coisas diferentes: **(a)** aplicar a mesma transformação aos
dois conjuntos; **(b)** medir o risco contra a função de regressão verdadeira, e
não contra o $y$ observado, que tem ruído; **(c)** prever no conjunto que está na
escala certa.

In [ ]:
rng = np.random.default_rng(2026)
n, p = 300, 5


def alvo(M):
    return 2.0 * M[:, 0] - M[:, 1] + 0.5 * M[:, 2] * M[:, 3]


X = rng.normal(size=(n, p))
y = alvo(X) + rng.normal(0, 0.5, size=n)
X_te = rng.normal(size=(2000, p))
r_te = alvo(X_te)

# a coluna 0 chega em outra unidade: 500 vezes maior
escala = np.ones(p)
escala[0] = 500.0
Xe, Xe_te = X * escala, X_te * escala          # (a) a MESMA escala nos dois

metodos = {
    "MQO": skl.LinearRegression(),
    "Ridge (alpha=50)": skl.Ridge(alpha=50.0),
    "arvore": DecisionTreeRegressor(max_depth=5, random_state=0),
    "KNN (k=10)": KNeighborsRegressor(n_neighbors=10),
    "Lasso (alpha=0,1)": skl.Lasso(alpha=0.1),
}

linhas = []
for nome, m in metodos.items():
    # o risco e' medido contra a regressao verdadeira, nao contra y ruidoso
    a = np.mean((m.fit(X, y).predict(X_te) - r_te) ** 2)        # (b)
    b = np.mean((m.fit(Xe, y).predict(Xe_te) - r_te) ** 2)      # (c)
    linhas.append({"metodo": nome, "escala original": a, "coluna 0 x500": b,
                   "mudou?": "sim" if abs(b - a) > 1e-6 * max(a, 1) else "nao"})

pd.DataFrame(linhas).set_index("metodo").round(4)

A tabela deve sair assim:

| método | escala original | coluna 0 x500 | mudou? |
|---|---|---|---|
| MQO | 0,2762 | 0,2762 | não |
| Ridge (alpha=50) | 0,4056 | 0,3123 | sim |
| árvore | 0,9384 | 0,9384 | não |
| KNN (k=10) | 0,8668 | 1,4527 | sim |
| Lasso (alpha=0,1) | 0,2990 | 0,2888 | sim |

**Dois ficaram parados.** O MQO é equivariante por reescala: se a coluna vira
$500\,x_j$, basta o coeficiente virar $\beta_j/500$ para o produto ficar igual, e é
isso que os mínimos quadrados encontram. A árvore nunca soma colunas — ela só
pergunta "$x_j \le t$?", e multiplicar a coluna multiplica o corte junto, separando
as mesmas observações. Nos dois casos os quatro dígitos são idênticos, não
parecidos.

**Três se mexeram.** Ridge e Lasso penalizam somas sobre colunas
($\sum_j\beta_j^2$ e $\sum_j|\beta_j|$), e somas assim só fazem sentido com
unidades comparáveis. O KNN é o caso mais dramático, com $+67\%$ de risco: a
distância euclidiana é $\sum_j (x_{ij}-x_{kj})^2$, então uma coluna 500 vezes maior
produz diferenças 500 vezes maiores e quadrados 250 mil vezes maiores. Ela passa a
decidir sozinha quem é vizinho de quem.

**E por que Ridge e Lasso melhoraram?** Essa é a parte que merece cuidado. A coluna
0 é a de maior efeito verdadeiro no `alvo`, e deixá-la 500 vezes maior fez o
coeficiente dela ficar 500 vezes menor — quase isento da penalidade. Isentar de
penalidade justamente a coluna que mais importa calhou de ajudar.

O que não é um argumento contra padronizar. É o argumento central **a favor**: quem
decidiu regularizar menos aquela coluna e mais as outras não foi você, foi a unidade
em que o dado chegou. O bloco a seguir explora isso.

> **Sua vez.** No exercício acima, a coluna reescalada foi a 0 — a de maior efeito
> verdadeiro. Refaça a comparação reescalando a **coluna 4**, que não aparece no
> `alvo` e portanto não tem efeito nenhum sobre a resposta.
>
> Antes de rodar, preveja: o Lasso e a Ridge vão melhorar de novo, ou vão piorar?
> Pense no que acontece quando é uma coluna **irrelevante** que fica quase isenta da
> penalidade — ela ganha liberdade para receber um coeficiente grande sem pagar por
> isso.

---
## Exercício 2 — o custo de padronizar fora da dobra

Padronizar exige **estimar** duas coisas por coluna: a média e o desvio. Como toda
estimativa, elas dependem de quais observações entraram na conta.

Se você ajustar o `StandardScaler` no conjunto todo antes de validar, as
estatísticas que o treino de cada dobra usa terão sido calculadas com a ajuda das
observações de validação daquela mesma dobra. Isso é vazamento: a validação cruzada
deixa de medir desempenho em dados novos.

O aviso está em todo livro. O que este exercício faz é pôr um número nele, no
cenário mais favorável possível ao alarme — sinal de verdade e oito colunas cujas
escalas vão de $0{,}01$ a $200$.

**Antes de rodar: quanto você acha que o $R^2$ vai inflar?** Anote o palpite; ele
faz parte do exercício.

As lacunas são **(a)** em que dados o scaler errado aprende, **(b)** qual peça entra
no tubo, e **(c)** a conta da diferença entre as duas condutas.

In [ ]:
n_e, REPETICOES = 60, 20
escalas = np.array([1, 50, 0.01, 5, 1, 200, 0.1, 2])
coeficientes = np.r_[1.5, 0.02, 80, 0.3, -1.0, 0.005, 10, 0.4]
cv = skm.KFold(5, shuffle=True, random_state=0)

fora, dentro = [], []
rng = np.random.default_rng(7)
for _ in range(REPETICOES):
    Xp = rng.normal(size=(n_e, 8)) * escalas
    yp = Xp @ coeficientes + rng.normal(0, 1, n_e)

    # ERRADO: o scaler aprende media e desvio no conjunto TODO
    esc = StandardScaler().fit(Xp)                             # (a)
    fora.append(skm.cross_val_score(skl.Ridge(alpha=1.0),
                                    esc.transform(Xp), yp,
                                    cv=cv, scoring="r2").mean())

    # CERTO: o scaler e uma etapa do pipeline, reajustada em cada dobra
    tubo = Pipeline([("escala", StandardScaler()),             # (b)
                     ("ridge", skl.Ridge(alpha=1.0))])
    dentro.append(skm.cross_val_score(tubo, Xp, yp, cv=cv, scoring="r2").mean())

print(f"escala FORA da dobra  : R2 = {np.mean(fora):.4f}")
print(f"escala DENTRO do tubo : R2 = {np.mean(dentro):.4f}")
print(f"diferenca             : {abs(np.mean(fora) - np.mean(dentro)):.2e}")   # (c)

Deve imprimir `escala FORA da dobra  : R2 = 0.8594`,
`escala DENTRO do tubo : R2 = 0.8593` e `diferenca: 1.11e-04`.

Se o seu palpite foi bem maior que isso, você está em boa companhia — é o que quase
todo mundo espera.

Os dois valores coincidem até a terceira casa decimal, e a diferença é da ordem de
$10^{-4}$. Isso no cenário que montamos de propósito para exagerar o problema.

**Por que tão pouco?** Porque o `StandardScaler` expõe muito pouca coisa: dois
números por coluna, cada um calculado sobre dezenas de observações. Trocar a média
de 48 observações pela média de 60 desloca o valor por algo da ordem de
$1/\sqrt n$ — e desloca todas as observações do mesmo jeito, **sem olhar o $y$ de
nenhuma delas**. Não há canal por onde o modelo possa aprender algo específico sobre
as observações de validação.

É o critério dos itens *leves* do Exercício 1 da Lista Teórica 06: o que separa
vazamento grave de leve não é quanto a etapa aprende dos dados, é se ela **olha a
resposta**.

E, apesar do número pequeno, a conduta não muda: ponha no `Pipeline`. Não porque
este erro seja caro, mas porque o acerto é de graça. O que a medição muda é **onde
gastar vigilância** — numa revisão de código, a pergunta cara é "como as covariáveis
foram escolhidas?", não "o scaler está no lugar certo?".

---
## Exercício 3 — o `Pipeline` inteiro, e o `GridSearchCV`

Feita a parte do diagnóstico, monte agora o que se deve fazer.

Um `Pipeline` encadeia o `StandardScaler` e o estimador num objeto só, e a promessa
dele é bem específica: **o conjunto de teste nunca entra em nenhum `fit`**. Não por
disciplina sua, mas por construção.

Promessa se verifica. A célula abaixo monta o tubo, ajusta, e depois pergunta ao
scaler de dentro dele qual média ele guardou — comparando com a média do treino e
com a do conjunto completo. Só uma das duas pode bater; antes de rodar, decida qual.

As lacunas são **(a)** a primeira etapa do tubo e **(b)** em quais dados o tubo é
ajustado.

In [ ]:
X_reg, y_reg = make_regression(n_samples=600, n_features=60, n_informative=8,
                              noise=1.0, random_state=2026)
X_tr, X_te2, y_tr, y_te2 = skm.train_test_split(X_reg, y_reg, test_size=0.3,
                                                random_state=2026)

tubo = Pipeline([("escala", StandardScaler()),                 # (a)
                 ("lasso", skl.Lasso())])
tubo.fit(X_tr, y_tr)                                           # (b) so o treino

media_aprendida = tubo.named_steps["escala"].mean_

print("bate com a media do TREINO?        ",
      np.allclose(media_aprendida, X_tr.mean(axis=0)))
print("bate com a media do conjunto TODO? ",
      np.allclose(media_aprendida, X_reg.mean(axis=0)))

Deve imprimir `True` e depois `False`, nessa ordem.

É a promessa verificada. Quando você chamou `tubo.fit(X_tr, y_tr)`, o
`StandardScaler` fez `fit_transform` no treino e só nele: a média que ele guardou é
a das 420 observações de treino, não a das 600. As outras 180 não participaram de
nada — e não teriam como participar, porque você nunca as passou para o `fit`.

O que isso significa na prática: sem o `Pipeline`, manter essa separação depende de
você lembrar, toda vez, de ajustar o scaler só no treino e depois aplicar *aquele
mesmo* scaler ao teste. Com ele, a única forma de errar seria passar os dados
errados para o `fit` — o que é bem mais difícil de fazer sem perceber.

Agora o ganho que justifica tudo: entregar o **tubo inteiro** ao `GridSearchCV`, em
vez de entregar só o Lasso.

Por que isso importa? Escolher o `alpha` por validação cruzada significa ajustar o
modelo $8 \times 5 = 40$ vezes. Se o objeto buscado for o tubo inteiro, a
padronização é refeita em cada um desses 40 ajustes, com as estatísticas da dobra
certa. Se você tivesse padronizado antes e passado só o Lasso, os 40 usariam as
mesmas estatísticas — calculadas com a ajuda de dados que deveriam ser novos.

Falta a sintaxe. Para alcançar um parâmetro que mora *dentro* de uma etapa,
escreve-se `nomeDaEtapa__parametro`, com **dois** sublinhados. Leia da esquerda para
a direita: o nome que você deu à etapa, depois `__` para descer um nível, depois o
parâmetro.

As lacunas são **(a)** a chave da grade, **(b)** em quais dados a busca roda, e
**(c)** em quais dados se mede no fim.

In [ ]:
grade = {"lasso__alpha": [0.001, 0.01, 0.1, 1, 5, 10, 50, 100]}     # (a)

busca = skm.GridSearchCV(tubo, grade, cv=5,
                         scoring="neg_mean_squared_error").fit(X_tr, y_tr)   # (b)

print(f"melhor alpha          : {busca.best_params_['lasso__alpha']:g}")
print(f"combinacoes avaliadas : {len(busca.cv_results_['mean_test_score'])}")
print(f"EQM de CV             : {-busca.best_score_:.4f}")
print(f"EQM no teste          : {mean_squared_error(y_te2, busca.predict(X_te2)):.4f}")   # (c)
print(f"coeficientes nao nulos: {(busca.best_estimator_.named_steps['lasso'].coef_ != 0).sum()} de 60")

Deve imprimir `melhor alpha : 0.1`, `combinacoes avaliadas : 8`,
`EQM de CV : 1.2113`, `EQM no teste : 1.0158` e `coeficientes nao nulos: 9 de 60`.

**A chave é `lasso__alpha`.** Sem os dois sublinhados não haveria como dizer ao
`GridSearchCV` onde o parâmetro mora. Se você quisesse variar também algo do scaler,
a chave seria `escala__with_mean`; se houvesse um `ColumnTransformer` no meio, os
níveis se encadeariam — `prep__num__imp__strategy`.

**O Lasso guardou 9 coeficientes de 60**, e os dados foram gerados com 8 colunas
informativas. Ele reencontrou quase exatamente a esparsidade certa, e isso só foi
possível porque as 60 colunas chegaram até ele na mesma escala. Sem o
`StandardScaler` uma linha acima, a penalidade $\ell_1$ estaria comparando
coeficientes em unidades diferentes, e a escolha de quais zerar seria arbitrária.

**O erro de teste ($1{,}0158$) saiu menor que o de validação cruzada ($1{,}2113$)**,
e isso surpreende quem lembra da regra "quem escolheu não pode reportar".

A regra continua valendo. Ela diz que o `best_score_` **tende** a ser otimista, por
ser o mínimo de várias estimativas ruidosas — e "tende a" não é "sempre". Aqui foram
apenas **oito** combinações, então o otimismo da seleção é pequeno; e o conjunto de
teste tem só 180 observações, então ele próprio carrega ruído. Neste sorteio, o
ruído do teste falou mais alto que o otimismo da seleção.

A conduta, mesmo assim, não muda: você reporta o número do teste, não o da validação
cruzada. O que a regra garante é a tendência ao longo de muitas amostras, não o
resultado de uma.

---
## O que ficou

| Pergunta | Resposta que você mediu |
|---|---|
| Quem muda quando a régua muda? | quem soma colunas: Ridge, Lasso e KNN. MQO e árvore, não |
| Qual é o mais frágil? | o KNN: $+67\%$ de risco com uma única coluna reescalada |
| Quanto custa padronizar fora da dobra? | $0{,}8594$ contra $0{,}8593$ — pouco. Mas corrigir custa zero |
| O `Pipeline` cumpre o que promete? | cumpre: o scaler bate com o treino e não com o conjunto todo |
| Como chego a um parâmetro de dentro dele? | `lasso__alpha`, com dois sublinhados |

**A seguir.** Fecha o Bloco I. A Aula 07 abre a classificação: a resposta deixa de
ser um número e passa a ser uma classe, o risco deixa de ser erro quadrático, e
aparece um classificador ótimo com nome próprio.